# Track 1 energy forecast — submission 007

Place the matching `model.pkl` beside this notebook. Run all cells to predict the input rows in their supplied order.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from threadpoolctl import threadpool_limits
TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")


In [ ]:
"""Row-local feature engineering + linear-model inference. This exact code is copied into the
prediction notebook, so it may only use numpy / pandas.

Robustness contract (the private test "may contain different observations and edge cases"):
every row gets a finite prediction even with missing / malformed / unseen values in ANY column.
"""
import numpy as np
import pandas as pd

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A", "LIB_A",
             "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]
TYPES = ["Administration", "Business", "Engineering", "LectureHall", "Library",
         "Residential", "Science", "Sports"]
BUILDING_TYPE = {"ADM_A": "Administration", "BUS_A": "Business", "BUS_B": "Business",
                 "ENG_A": "Engineering", "ENG_B": "Engineering", "LEC_A": "LectureHall",
                 "LIB_A": "Library", "RES_A": "Residential", "RES_B": "Residential",
                 "SCI_A": "Science", "SCI_B": "Science", "SPT_A": "Sports"}
NUMERIC = ["previous_usage", "occupancy", "temperature", "humidity"]
# physically plausible ranges; values outside are treated as sensor errors -> missing
VALID = {"previous_usage": (0, 1000), "occupancy": (0, 10000), "temperature": (-20, 60), "humidity": (0, 100)}


def _num(s):
    return pd.to_numeric(s, errors="coerce").astype(float)


def clean_input(df):
    """Sanitise raw columns; anything unusable becomes NaN (handled downstream)."""
    out = pd.DataFrame(index=df.index)
    get = lambda c: df[c] if c in df.columns else pd.Series(np.nan, index=df.index)
    out["building_id"] = get("building_id").astype(str).str.strip().str.upper()
    out["building_type"] = get("building_type").astype(str).str.strip()
    hour = _num(get("hour"))
    out["hour"] = hour.where(hour.between(0, 23)).round()
    month = _num(get("month"))
    out["month"] = month.where(month.between(1, 12)).round()
    day = get("day_of_week").astype(str).str.strip().str[:3].str.lower()
    out["dow"] = day.map({d[:3].lower(): i for i, d in enumerate(DAYS)}).astype(float)
    for c in NUMERIC:
        v = _num(get(c))
        lo, hi = VALID[c]
        out[c] = v.where(v.between(lo, hi))
    return out


def make_features(df):
    """Tabular features for tree models (used by the GBM baseline)."""
    c = clean_input(df)
    out = pd.DataFrame(index=df.index)
    out["building"] = pd.Categorical(c["building_id"], categories=BUILDINGS)
    out["btype"] = pd.Categorical(c["building_type"], categories=TYPES)
    out["hour"] = c["hour"]
    out["dow"] = c["dow"]
    out["weekend"] = (c["dow"] >= 5).astype(float).where(c["dow"].notna())
    out["month"] = c["month"]
    for col in ["temperature", "humidity", "occupancy", "previous_usage"]:
        out[col] = c[col]
    return out


def _soft_building(c):
    """n x 12 building weights: one-hot if known, else uniform over buildings of the given type,
    else uniform over all buildings (=> prediction is the average building's expectation)."""
    n = len(c)
    W = np.zeros((n, len(BUILDINGS)))
    bid = c["building_id"].to_numpy(); bt = c["building_type"].to_numpy()
    for i in range(n):
        if bid[i] in BUILDINGS:
            W[i, BUILDINGS.index(bid[i])] = 1.0
        else:
            same = [j for j, b in enumerate(BUILDINGS) if BUILDING_TYPE[b] == bt[i]]
            W[i, same if same else slice(None)] = 1.0 / (len(same) if same else len(BUILDINGS))
    return W


def _soft_onehot(v, k):
    v = np.asarray(v, dtype=float)
    out = np.full((len(v), k), 1.0 / k)
    ok = ~np.isnan(v)
    out[ok] = 0.0
    out[np.where(ok)[0], v[ok].astype(int)] = 1.0
    return out


DEFAULT_SPEC = {"hours": "type", "weekend": "building",
                "slopes": {"previous_usage": "shared", "occupancy": "building",
                           "temperature": "building", "humidity": "shared"},
                "temp2": True, "season": True}


def linear_design(c, drop=(), spec=DEFAULT_SPEC):
    """Design matrix from a clean_input() frame. `drop` = numeric columns unavailable for this
    missingness-pattern sub-model. Categorical unknowns enter as expectations (soft one-hots)."""
    if spec.get("missing_spec") and set(drop) & {"occupancy", "previous_usage"}:
        # sub-models without occupancy/previous_usage must rebuild them from time x building -> richer design
        spec = {**spec, **spec["missing_spec"]}
    n = len(c)
    B = _soft_building(c)
    T = B @ np.array([[BUILDING_TYPE[b] == t for t in TYPES] for b in BUILDINGS], dtype=float)
    G = {"building": B, "type": T, "shared": np.ones((n, 1))}
    K = spec.get("hour_fourier", 0)
    if K:  # smooth daily profile: K harmonics (missing hour -> 0 = daily average)
        hh = c["hour"].to_numpy()
        H = np.concatenate([f(2 * np.pi * k * hh / 24)[:, None] for k in range(1, K + 1) for f in (np.sin, np.cos)], 1)
        H = np.nan_to_num(H, nan=0.0)
    else:
        H = _soft_onehot(c["hour"], 24)
    dow = c["dow"].to_numpy()
    we = np.where(np.isnan(dow), 2 / 7, (dow >= 5).astype(float))[:, None]
    cols = [B, (G[spec["hours"]][:, :, None] * H[:, None, :]).reshape(n, -1), G[spec["weekend"]] * we]
    if spec.get("hour_dev_scale"):  # per-hour deviations from the smooth profile, shrunk by a heavier
        # effective ridge penalty (alpha / scale^2) -> captures schedule steps (e.g. residential 08:00 / 21:00)
        Hd = _soft_onehot(c["hour"], 24) * spec["hour_dev_scale"]
        Gd = G[spec.get("hour_dev_group", "type")]
        cols.append((Gd[:, :, None] * Hd[:, None, :]).reshape(n, -1))
    if spec.get("weekend_hours"):  # separate daily shape on weekends
        Gw = G[spec["weekend_hours"]]
        cols.append((Gw[:, :, None] * (H * we)[:, None, :]).reshape(n, -1))
    for col in NUMERIC:
        if col not in drop:
            cols.append(G[spec["slopes"][col]] * c[col].to_numpy()[:, None])
    # extra shrinkable interactions: (numeric col, group, scale); effective penalty alpha/scale^2
    if spec.get("interactions"):
        hr = c["hour"].to_numpy()
        bands = np.stack([(hr <= 6), (hr >= 7) & (hr <= 11), (hr >= 12) & (hr <= 17), (hr >= 18)], 1).astype(float)
        bands[np.isnan(hr)] = 0.25
        we2 = np.concatenate([1 - we, we], 1)
        GX = {**G, "hourband": bands, "weekend": we2}
        for col, grp, scale in spec["interactions"]:
            if col in drop:
                continue
            if col == "temp_hinge":
                v = np.stack([np.maximum(c["temperature"].to_numpy() - k, 0) for k in (26.0, 28.0, 30.0)], 1)
                if "temperature" in drop:
                    continue
                cols.append((GX[grp][:, :, None] * v[:, None, :]).reshape(n, -1) * scale)
                continue
            cols.append(GX[grp] * c[col].to_numpy()[:, None] * scale)
    if spec.get("season", True):
        m = c["month"].to_numpy()
        sc = np.stack([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)], 1)
        cols.append(np.nan_to_num(sc, nan=0.0))
    if spec.get("temp2", True) and "temperature" not in drop:
        cols.append(((c["temperature"].to_numpy() - 28.0) ** 2)[:, None])
    return np.concatenate(cols, 1)


def pattern_keys(c):
    miss = c[NUMERIC].isna().to_numpy()
    return [tuple(col for col, mm in zip(NUMERIC, row) if mm) for row in miss]


def predict_linear(art, df):
    """Pattern-submodel linear prediction: each row uses the model fitted without its missing
    numeric columns. art = {"coefs": {pattern: (coef, intercept)}, "spec": ..., "clip": (lo, hi)}."""
    c = clean_input(df)
    keys = pattern_keys(c)
    preds = np.empty(len(c))
    for k in set(keys):
        idx = np.array([kk == k for kk in keys])
        coef, icpt = art["coefs"][k]
        preds[idx] = linear_design(c[idx], drop=set(k), spec=art["spec"]) @ coef + icpt
    return np.clip(preds, *art["clip"])


def _dev_keys(F, level):
    cols = [F[x].cat.codes.to_numpy() if hasattr(F[x], "cat") else F[x].to_numpy() for x in level]
    return list(zip(*cols))

def add_dev(F, tables, levels, skip):
    """Append deviation features (fixed order: level, then column) - identical in training and inference."""
    G = F.copy()
    for i, level in enumerate(levels):
        keys = _dev_keys(F, level)
        for c in ["occupancy", "previous_usage", "temperature", "humidity"]:
            if c != skip:
                mu = np.array([tables[i][c].get(k, np.nan) for k in keys], dtype=float)
                G[f"dev{i}_{c}"] = F[c].to_numpy(dtype=float) - mu
    return G

LEVELS = [['building', 'hour'], ['building']]
NUMS = ['occupancy', 'previous_usage', 'temperature', 'humidity']
V4 = {'hours': 'type', 'weekend': 'building', 'slopes': {'previous_usage': 'shared', 'occupancy': 'building', 'temperature': 'building', 'humidity': 'shared'}, 'temp2': True, 'season': True, 'hour_fourier': 3, 'weekend_hours': 'shared', 'missing_spec': {'hour_fourier': 6, 'weekend_hours': 'type'}, 'hour_dev_scale': 0.25}

def context_design(rows):
    F=make_features(rows); c=clean_input(rows); n=len(c)
    B=_soft_building(c)
    T=B@np.array([[BUILDING_TYPE[b]==t for t in TYPES] for b in BUILDINGS],float)
    h=F.hour.fillna(12).to_numpy(); we=F.weekend.fillna(2/7).to_numpy()[:,None]
    H=np.eye(24)[h.astype(int)]
    phase=2*np.pi*F.month.fillna(6).to_numpy()/12
    annual=np.column_stack([np.sin(phase),np.cos(phase),np.sin(2*phase),np.cos(2*phase)])
    daily=np.column_stack([np.sin(2*np.pi*h/24),np.cos(2*np.pi*h/24)])
    return np.column_stack([B, (T[:,:,None]*H[:,None,:]).reshape(n,-1),
        .2*(B[:,:,None]*H[:,None,:]).reshape(n,-1), B*we,
        .5*(T[:,:,None]*(H*we)[:,None,:]).reshape(n,-1), annual,
        .3*(T[:,:,None]*annual[:,None,:]).reshape(n,-1),
        .5*(annual[:,:,None]*daily[:,None,:]).reshape(n,-1)])


def profile_values(rows, profiles):
    D=context_design(rows); F=make_features(rows); out={}
    for col,p in profiles.items():
        mu=D@p['coef']+p['intercept']
        sd=F.building.cat.codes.map(p['scales']).fillna(p['default_scale']).to_numpy(float)
        out[col]=(mu,sd)
    return out


def sensor_features(F, tables, skip, profiles=None, values=None, cyclic=False):
    G=add_dev(F,tables,LEVELS,skip).drop(columns=[skip])
    if cyclic:
        for col,period in [('hour',24),('month',12)]:
            for k in [1,2]:
                G[f'{col}_sin{k}']=np.sin(2*np.pi*k*F[col]/period)
                G[f'{col}_cos{k}']=np.cos(2*np.pi*k*F[col]/period)
    if profiles is not None:
        for col in NUMS:
            mu,sd=values[col]
            G['profile_'+col]=mu
            G['scale_'+col]=sd
            if col!=skip:
                G['profile_dev_'+col]=(F[col].to_numpy(float)-mu)/sd
    return G


def predict_sensor(artifact, rows, col, domain=1):
    F=make_features(rows); F['dom']=domain
    profiles=artifact['profiles']; values=profile_values(rows,profiles) if profiles else None
    G=sensor_features(F,artifact['tables'],col,profiles,values,artifact['config'].get('cyclic',False))
    p=np.mean([lgb.Booster(model_str=s).predict(G,num_threads=4) for s in artifact['models'][col]],axis=0)
    if artifact['config'].get('residual'): p=values[col][0]+values[col][1]*p
    return p


def energy_design(frame, config):
    c = clean_input(frame)
    spec = dict(V4)
    spec.update(config.get('spec', {}))
    D = linear_design(c, spec=spec)
    B = _soft_building(c)
    T = B @ np.array([[BUILDING_TYPE[b] == t for t in TYPES] for b in BUILDINGS], float)
    group = {'building': B, 'type': T, 'shared': np.ones((len(c), 1))}
    if config.get('steps'):
        h = c.hour.to_numpy()
        residential = T[:, TYPES.index('Residential')]
        # Generator-like residential switch schedule; Fourier alone smooths the edges.
        cols = [np.where(np.isfinite(h), ((h >= 8) & (h < 21)).astype(float), 13/24) * residential,
                np.where(np.isfinite(h), ((h >= 21) | (h < 8)).astype(float), 11/24) * residential]
        D = np.column_stack([D, *cols])
    if config.get('season_group'):
        phase = 2*np.pi*c.month.to_numpy()/12
        wave = np.nan_to_num(np.column_stack([np.sin(phase), np.cos(phase)]))
        D = np.column_stack([D, (group[config['season_group']][:,:,None]*wave[:,None,:]).reshape(len(c), -1)])
    if config.get('cooling'):
        D = np.column_stack([D, T*np.maximum(c.temperature.to_numpy()[:,None]-24, 0)])
    return D


def predict_energy(artifact, frame):
    return energy_design(frame, artifact['config']) @ artifact['coef'] + artifact['intercept']


def fill_sensor_plan(sensors, plan, frame, domain=1, clip_predictions=True):
    original=frame.reset_index(drop=True)
    out=original.copy(); clean=clean_input(out)
    for col in NUMS:
        raw=clean[col].to_numpy(float); miss=~np.isfinite(raw); out[col]=raw
        if miss.any():
            pred=sum(weight*predict_sensor(sensors[name],original.iloc[np.flatnonzero(miss)],col,domain)
                     for name,weight in plan[col])
            # Each sensor is predicted from the original observed row, never
            # from another sensor's imputed value.
            lo,hi=VALID[col]
            out.loc[np.flatnonzero(miss),col]=np.clip(pred,lo,hi) if clip_predictions else pred
    return out


def predict_model(artifact,frame):
    if len(frame)==0: return np.empty(0,dtype=float)
    with threadpool_limits(limits=4),np.errstate(all='ignore'):
        filled=fill_sensor_plan(artifact['sensors'],artifact['configuration']['sensor_plan'],frame)
        p=predict_energy(artifact['energy'],filled)
        bad=~np.isfinite(p)
        if bad.any(): p[bad]=predict_linear(artifact['fallback'],frame.iloc[np.flatnonzero(bad)])
    p=np.clip(np.asarray(p,float),0,400)
    if p.shape!=(len(frame),) or not np.isfinite(p).all(): raise ValueError('Expected one finite prediction per row')
    return p



In [ ]:
artifact = joblib.load("model.pkl")
test = pd.read_csv(TEST_PATH)
preds = predict_model(artifact, test)


In [ ]:
output = pd.DataFrame({"prediction": preds})
if "id" in test.columns:
    output.insert(0, "id", test["id"].to_numpy())
output.to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")
